In [1]:
# CalFireRiskIQ — Feature Engineering

#Goal: Prepare clean numeric features and balanced classes ready for ML model.

### Steps:
#1. Load merged data
#2. Drop unnecessary columns
#3. Encode categorical variables
#4. Define X and y
#5. Train/test split
#6. Apply SMOTE
#7. Save final datasets

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from imblearn.over_sampling import SMOTE
import warnings
warnings.filterwarnings('ignore')

# Load merged dataset
df = pd.read_csv('../data/processed/merged_calfire.csv')

print(f" Loaded: {df.shape[0]:,} rows, {df.shape[1]} columns")
display(df.head(3))

 Loaded: 228,305 rows, 25 columns


,fire_name,incident_date,county,damage_level,structure_category,structure_type,roof_construction,eaves,vent_screen,exterior_siding,...,fire_year,risk_level,gis_acres,cause,agency,fire_duration_days,area_burned,homes_destroyed,financial_loss_million,fatalities
0,QUAIL,6/6/2020 12:00:00 AM,SOLANO,NO DAMAGE,Single Residence,Single Family Residence Multi Story,Asphalt,Unenclosed,"Mesh Screen <= 1/8""""",Wood,...,2020,Low,1841.00000,14.0,CDF,12.0,0.0,0.0,0.0,0.0
1,QUAIL,6/6/2020 12:00:00 AM,SOLANO,NO DAMAGE,Single Residence,Single Family Residence Multi Story,Asphalt,Unenclosed,"Mesh Screen <= 1/8""""",Wood,...,2020,Low,12.85847,5.0,CCO,0.0,0.0,0.0,0.0,0.0
2,QUAIL,6/6/2020 12:00:00 AM,SOLANO,AFFECTED (>0-10%),Single Residence,Single Family Residence Single Story,Asphalt,Unenclosed,"Mesh Screen <= 1/8""""",Wood,...,2020,Low,1841.00000,14.0,CDF,12.0,0.0,0.0,0.0,0.0


In [3]:
#Drop Unnecessary Columns

In [4]:
# Columns to drop and why:
# fire_name     → too many unique values, not a useful pattern
# incident_date → already have fire_year extracted
# damage_level  → this is the raw source of target, not a feature
# agency        → administrative, not predictive

drop_cols = ['fire_name', 'incident_date', 'damage_level', 'agency']

df_model = df.drop(columns=drop_cols)

print(f" Dropped {len(drop_cols)} columns")
print(f"Remaining columns ({df_model.shape[1]}):")
print(df_model.columns.tolist())

 Dropped 4 columns
Remaining columns (21):
['county', 'structure_category', 'structure_type', 'roof_construction', 'eaves', 'vent_screen', 'exterior_siding', 'window_pane', 'year_built', 'assessed_value', 'latitude', 'longitude', 'fire_year', 'risk_level', 'gis_acres', 'cause', 'fire_duration_days', 'area_burned', 'homes_destroyed', 'financial_loss_million', 'fatalities']


In [5]:
#Encode Categorical Variables

In [6]:
# Remove rows with no fire data (unmatched fires)
df_model = df_model[df_model['gis_acres'] > 0]
print(f"Rows after removing unmatched: {len(df_model):,}")

# Check which columns are text/categorical
cat_cols = df_model.select_dtypes(include='object').columns.tolist()
cat_cols = [c for c in cat_cols if c != 'risk_level']  # exclude target

print(f"Categorical columns to encode: {cat_cols}")

# Fill NaN in categorical columns BEFORE encoding
for col in cat_cols:
    df_model[col] = df_model[col].fillna('Unknown')

# One Hot Encoding
df_encoded = pd.get_dummies(df_model, columns=cat_cols, drop_first=True)

# Fill any remaining NaNs after encoding
df_encoded = df_encoded.fillna(0)

print(f"\n Encoding complete")
print(f"Columns before encoding : {df_model.shape[1]}")
print(f"Columns after encoding  : {df_encoded.shape[1]}")
print(f"NaNs remaining: {df_encoded.isnull().sum().sum()}")

Rows after removing unmatched: 217,790
Categorical columns to encode: ['county', 'structure_category', 'structure_type', 'roof_construction', 'eaves', 'vent_screen', 'exterior_siding', 'window_pane']

 Encoding complete
Columns before encoding : 21
Columns after encoding  : 131
NaNs remaining: 0


In [7]:
#Define X (Features) and y (Target)

In [8]:
# y = target variable (what we're predicting)
y = df_encoded['risk_level']

# X = everything else (features the model learns from)
X = df_encoded.drop(columns=['risk_level'])

print(f"X shape : {X.shape}  ← features matrix")
print(f"y shape : {y.shape}  ← target vector")
print(f"\ny (risk_level) distribution:")
print(y.value_counts())
print(f"\nX columns ({len(X.columns)}):")
print(X.columns.tolist())

X shape : (217790, 130)  ← features matrix
y shape : (217790,)  ← target vector

y (risk_level) distribution:
risk_level
High      132841
Low        81710
Medium      3239
Name: count, dtype: int64

X columns (130):
['year_built', 'assessed_value', 'latitude', 'longitude', 'fire_year', 'gis_acres', 'cause', 'fire_duration_days', 'area_burned', 'homes_destroyed', 'financial_loss_million', 'fatalities', 'county_ALPINE', 'county_AMADOR', 'county_BUTTE', 'county_CALAVERAS', 'county_COLUSA', 'county_CONTRA COSTA', 'county_EL DORADO', 'county_FRESNO', 'county_GLENN', 'county_HUMBOLDT', 'county_INYO', 'county_KERN', 'county_KINGS', 'county_LAKE', 'county_LASSEN', 'county_LOS ANGELES', 'county_MADERA', 'county_MARIPOSA', 'county_MENDOCINO', 'county_MONO', 'county_MONTEREY', 'county_NAPA', 'county_NEVADA', 'county_ORANGE', 'county_PLACER', 'county_PLUMAS', 'county_RIVERSIDE', 'county_SACRAMENTO', 'county_SAN BENITO', 'county_SAN BERNARDINO', 'county_SAN DIEGO', 'county_SAN JOAQUIN', 'county_SAN

In [9]:
#Train / Test Split

In [10]:
# Split: 80% training, 20% testing
# random_state=42 → same split every time (reproducible)
# stratify=y → keeps same risk level proportions in both splits

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f" Train/Test Split complete")
print(f"X_train : {X_train.shape}")
print(f"X_test  : {X_test.shape}")
print(f"\ny_train distribution:")
print(y_train.value_counts())
print(f"\ny_test distribution:")
print(y_test.value_counts())

 Train/Test Split complete
X_train : (174232, 130)
X_test  : (43558, 130)

y_train distribution:
risk_level
High      106273
Low        65368
Medium      2591
Name: count, dtype: int64

y_test distribution:
risk_level
High      26568
Low       16342
Medium      648
Name: count, dtype: int64


In [11]:
# Fill NaN in cause columns with 0 (means 'not this cause')
# After one-hot encoding, cause becomes cause_Human, cause_Lightning etc.
# NaN in these = fire had no cause data = just set to 0

cause_cols = [col for col in X_train.columns if col.startswith('cause_')]
print(f"Cause columns found: {cause_cols}")

X_train[cause_cols] = X_train[cause_cols].fillna(0)
X_test[cause_cols]  = X_test[cause_cols].fillna(0)

# Confirm
print(f"\nNaNs in X_train: {X_train.isnull().sum().sum()}")
print(f"NaNs in X_test:  {X_test.isnull().sum().sum()}")

Cause columns found: []

NaNs in X_train: 0
NaNs in X_test:  0


In [12]:
#Apply SMOTE (Fix Class Imbalance)

In [13]:
import os
os.environ["LOKY_MAX_CPU_COUNT"] = "1"

from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)
X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)

print(f"SMOTE applied to training data only")
print(f"\nBefore SMOTE:")
print(y_train.value_counts())
print(f"\nAfter SMOTE:")
print(pd.Series(y_train_bal).value_counts())
print(f"\nX_train shape after SMOTE: {X_train_bal.shape}")

SMOTE applied to training data only

Before SMOTE:
risk_level
High      106273
Low        65368
Medium      2591
Name: count, dtype: int64

After SMOTE:
risk_level
Low       106273
High      106273
Medium    106273
Name: count, dtype: int64

X_train shape after SMOTE: (318819, 130)


In [14]:
#Save Final Datasets

In [15]:
import os
os.makedirs('../data/processed', exist_ok=True)

# Save all 4 splits
X_train_bal_df = pd.DataFrame(X_train_bal, columns=X.columns)
y_train_bal_df = pd.Series(y_train_bal, name='risk_level')

X_train_bal_df.to_csv('../data/processed/X_train.csv', index=False)
y_train_bal_df.to_csv('../data/processed/y_train.csv', index=False)

pd.DataFrame(X_test, columns=X.columns).to_csv('../data/processed/X_test.csv', index=False)
pd.Series(y_test, name='risk_level').to_csv('../data/processed/y_test.csv', index=False)

print(" All 4 files saved to data/processed/")
print("   X_train.csv  ← balanced training features")
print("   y_train.csv  ← balanced training labels")
print("   X_test.csv   ← test features (real data)")
print("   y_test.csv   ← test labels (real data)")

 All 4 files saved to data/processed/
   X_train.csv  ← balanced training features
   y_train.csv  ← balanced training labels
   X_test.csv   ← test features (real data)
   y_test.csv   ← test labels (real data)
